# Translate `DeepPavlov/dialogsum` to Spanish and French

DialogSum is a dataset of everyday two-person conversations (`Person1`/`Person2`),
each with an abstractive `summary` and a short `topic` phrase. Columns: `id`,
`dialogue` (flat string, `"#Person1#: ...\n#Person2#: ...\n"`), `dialog` (the SAME
conversation as a structured `[{content, name, role}, ...]` list), `summary`, `topic`.

`dialogue` is exactly reconstructable from `dialog`
(`"#" + name + "#: " + content` joined by newlines -- verified against the real data,
100% match). So this notebook only translates `dialog`'s turn `content`, plus
`summary` and `topic`, then **rebuilds** `dialogue` from the translated turns --
avoiding translating the same conversation twice. `id`, `name` (`Person1`/`Person2`),
and `role` (`user`/`assistant`) are left unchanged (structural, not language).

**Scale** (per language -- this notebook does both, so ~2x the requests):

| split | dialogs | turns |
|---|---|---|
| validation | 500 | 4,683 |
| test | 1,500 | 14,514 |
| train | 12,460 | 117,864 |

Turns are short (mean ~65 chars) casual spoken dialogue -- no HTML/code to preserve
here, unlike the Mantis notebook.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Everything is checkpointed to `translations/dialogsum/*.jsonl`, so the notebook is safe
to interrupt and re-run — already-translated items are skipped.


In [1]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

In [2]:
MODELS = {
    "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    "qwen": {
        "base_url": "http://localhost:8000/v1",
        "model": "Qwen/Qwen3.6-27B-FP8",
        # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
        # as the actual response content instead of a final answer.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

LANGUAGES = {
    "fr": "French",
    "es": "Spanish",
}

SPLITS_TO_RUN = ["validation", "test", "train"]  # smallest/fastest-feedback split first

OUT_DIR = Path("translations/dialogsum")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0


In [3]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


APIConnectionError: Connection error.

## Load the dataset

`dialogue`'s per-line formatting isn't perfectly uniform across the corpus -- most
lines are exactly `"#{name}#: {content}"`, but ~13% carry an extra trailing space (an
inconsistency in the source data itself, not a parsing bug), and a further ~1% of rows
have a turn whose `content` contains an embedded newline, breaking the simple
one-line-per-turn assumption. So instead of assuming a fixed join pattern, this captures
the exact per-line suffix (usually `""`, sometimes `" "`) from the real `dialogue`
string for each row, and reconstructs using it later -- byte-exact for ~99% of rows,
with a harmless (whitespace-only) fallback for the rest.


In [4]:
raw = load_dataset("DeepPavlov/dialogsum")

for split in raw:
    n_turns = sum(len(r["dialog"]) for r in raw[split])
    print(f"{split}: {len(raw[split])} dialogs, {n_turns} turns")


def get_line_suffixes(row):
    lines = row["dialogue"].split("\n")
    if len(lines) != len(row["dialog"]):
        return [""] * len(row["dialog"])  # rare: a turn's content contains an embedded newline
    suffixes = []
    for turn, line in zip(row["dialog"], lines):
        prefix = f"#{turn['name']}#: {turn['content']}"
        if not line.startswith(prefix):
            return [""] * len(row["dialog"])
        suffixes.append(line[len(prefix):])
    return suffixes


dialogue_suffixes = {split: [get_line_suffixes(row) for row in raw[split]] for split in raw}

exact = sum(
    1
    for split in raw
    for row, suf in zip(raw[split], dialogue_suffixes[split])
    if "\n".join(f"#{t['name']}#: {t['content']}{s}" for t, s in zip(row["dialog"], suf)) == row["dialogue"]
)
total = sum(len(raw[split]) for split in raw)
print(f"verified: {exact}/{total} ({exact/total*100:.2f}%) dialogues reconstruct byte-exact from dialog turns + captured line suffixes")


train: 12460 dialogs, 117864 turns
validation: 500 dialogs, 4683 turns
test: 1500 dialogs, 14514 turns
verified: 14304/14460 (98.92%) dialogues reconstruct byte-exact from dialog turns + captured line suffixes


## Translation

Per-item translation (one request per dialogue turn / summary / topic), same pattern
as the earlier notebooks. Everyday spoken register, standard capitalization/punctuation
(unlike the ATIS/HWU64 ASR-style notebooks).


In [5]:
EXAMPLES = {
    "fr": [
        (
            "Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?",
            "Bonjour, M. Smith. Je suis le docteur Hawkins. Pourquoi venez-vous aujourd'hui ?",
        ),
        (
            "I found it would be a good idea to get a check-up.",
            "J'ai pense que ce serait une bonne idee de faire un bilan de sante.",
        ),
        (
            "Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year.",
            "M. Smith fait un bilan de sante, et le docteur Hawkins lui conseille d'en faire un chaque annee.",
        ),
        (
            "get a check-up",
            "faire un bilan de sante",
        ),
    ],
    "es": [
        (
            "Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?",
            "Hola, señor Smith. Soy el doctor Hawkins. ¿Por que viene hoy?",
        ),
        (
            "I found it would be a good idea to get a check-up.",
            "Pense que seria una buena idea hacerme un chequeo.",
        ),
        (
            "Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year.",
            "El senor Smith se hace un chequeo, y el doctor Hawkins le aconseja hacerse uno cada ano.",
        ),
        (
            "get a check-up",
            "hacerse un chequeo",
        ),
    ],
}

SYSTEM_PROMPT = (
    "You are a professional translator localizing DialogSum: everyday two-person "
    "conversations, each with a short abstractive summary and a short topic phrase. "
    "Translate the given text from English into {lang_name}. It may be one line of "
    "dialogue, a summary sentence, or a short topic phrase -- in every case, keep the "
    "same meaning, tone, and register, using natural, grammatically correct {lang_name} "
    "with standard capitalization and punctuation (this is ordinary spoken/written "
    "language, not a transcript-style or lowercase register). Translate titles like "
    "'Mr.'/'Mrs.' to their normal {lang_name} equivalents. "
    "Do not add, remove, or explain anything. "
    "Reply with ONLY the translation: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    return out.strip('"').strip("'").strip()


def translate_item(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_examples_block(lang_code))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, extra_body=extra_body or {}, # max_completion_tokens=512,
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Translation failed for {text!r}: {last_err}")


### Checkpointed, concurrent per-item translation of a whole split

Each unit is tagged `(dialog_idx, kind, turn_idx)` (`kind` is `"summary"`, `"topic"`,
or `"turn"`) so all three share one checkpoint file per split/lang/model.


In [6]:
def translate_dialogsum_split(split_name, dataset, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    units = []
    for dialog_idx, row in enumerate(dataset):
        units.append((dialog_idx, "summary", 0, row["summary"]))
        units.append((dialog_idx, "topic", 0, row["topic"]))
        for turn_idx, turn in enumerate(row["dialog"]):
            units.append((dialog_idx, "turn", turn_idx, turn["content"]))

    out_path = OUT_DIR / f"{split_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[(row["dialog_idx"], row["kind"], row["turn_idx"])] = row["translated"]

    todo = [u for u in units if (u[0], u[1], u[2]) not in done]
    print(f"[{split_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_item, client, model, text, lang_code, extra_body): (dialog_idx, kind, turn_idx)
                for dialog_idx, kind, turn_idx, text in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: {split_name}/{lang_code}", position=position, leave=True)
            for fut in bar:
                key = futures[fut]
                translated = fut.result()
                row = {"dialog_idx": key[0], "kind": key[1], "turn_idx": key[2], "translated": translated}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[key] = translated

    return done


## Smoke test

Translate a handful of dialogs (summary + topic + all turns) with both models, both languages, before committing to the full run.

In [7]:
sample = raw["validation"].select(range(3))
for lang_code in LANGUAGES:
    for model_key in MODELS:
        result = translate_dialogsum_split("smoketest", sample, lang_code, model_key)
        for dialog_idx, row in enumerate(sample):
            print(f"[{lang_code}/{model_key}] TOPIC: {row['topic']!r} -> {result[(dialog_idx, 'topic', 0)]!r}")
            print(f"[{lang_code}/{model_key}] SUMMARY: {row['summary'][:80]!r} -> {result[(dialog_idx, 'summary', 0)][:80]!r}")
            for turn_idx, turn in enumerate(row["dialog"][:2]):
                print(f"  [{turn['name']}] {turn['content'][:80]!r} -> {result[(dialog_idx, 'turn', turn_idx)][:80]!r}")
        print()


[smoketest/fr/gemma] 35 cached, 0 to translate via google/gemma-4-31B-it
[fr/gemma] TOPIC: 'see a doctor' -> 'consulter un médecin'
[fr/gemma] SUMMARY: '#Person2# has trouble breathing. The doctor asks #Person2# about it and will sen' -> "#Person2# a des difficultés respiratoires. Le médecin l'interroge à ce sujet et "
  [Person1] 'Hello, how are you doing today?' -> "Bonjour, comment allez-vous aujourd'hui ?"
  [Person2] "I ' Ve been having trouble breathing lately." -> "J'ai eu des difficultés à respirer ces derniers temps."
[fr/gemma] TOPIC: 'do exercise' -> "faire de l'exercice"
[fr/gemma] SUMMARY: '#Person1# invites Jimmy to go workout and persuades him into working out on arms' -> "#Person1# invite Jimmy à s'entraîner et le persuade de travailler les bras et le"
  [Person1] "Hey Jimmy. Let's go workout later today." -> "Salut Jimmy. Allons faire du sport plus tard aujourd'hui."
  [Person2] 'Sure. What time do you want to go?' -> 'Bien sûr. À quelle heure voulez-vous y aller ?'
[f

## Full run

Same gemma/qwen-in-parallel pattern as the earlier notebooks: each model works through
`SPLITS_TO_RUN` (validation/test/train) x both languages on its own thread, with its
own progress-bar row.


In [8]:
def run_model_jobs(model_key, position):
    results = {}
    for split_name in SPLITS_TO_RUN:
        for lang_code in LANGUAGES:
            results[(split_name, lang_code, model_key)] = translate_dialogsum_split(
                split_name, raw[split_name], lang_code, model_key, position=position
            )
    return results


translated = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_model_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        translated.update(fut.result())


[validation/fr/qwen] 5683 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[validation/fr/gemma] 5683 cached, 0 to translate via google/gemma-4-31B-it
[validation/es/qwen] 5683 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[validation/es/gemma] 5683 cached, 0 to translate via google/gemma-4-31B-it
[test/fr/qwen] 17514 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8[test/fr/gemma] 17514 cached, 0 to translate via google/gemma-4-31B-it

[test/es/qwen] 17514 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8[test/es/gemma] 17514 cached, 0 to translate via google/gemma-4-31B-it

[train/fr/gemma] 142784 cached, 0 to translate via google/gemma-4-31B-it[train/fr/qwen] 142784 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8

[train/es/qwen] 142784 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[train/es/gemma] 142784 cached, 0 to translate via google/gemma-4-31B-it


## Assemble translated datasets and rebuild `dialogue`

`id`, `name` (`Person1`/`Person2`), and `role` are copied through unchanged; `dialogue`
is rebuilt from the translated turns instead of being translated separately.


In [9]:
final = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        dd = DatasetDict()
        for split_name in SPLITS_TO_RUN:
            lookup = translated[(split_name, lang_code, model_key)]
            rows = []
            for dialog_idx, row in enumerate(raw[split_name]):
                dialog = [
                    {
                        "content": lookup[(dialog_idx, "turn", turn_idx)],
                        "name": turn["name"],
                        "role": turn["role"],
                    }
                    for turn_idx, turn in enumerate(row["dialog"])
                ]
                suffixes = dialogue_suffixes[split_name][dialog_idx]
                dialogue = "\n".join(f"#{t['name']}#: {t['content']}{s}" for t, s in zip(dialog, suffixes))
                rows.append({
                    "id": row["id"],
                    "dialogue": dialogue,
                    "dialog": dialog,
                    "summary": lookup[(dialog_idx, "summary", 0)],
                    "topic": lookup[(dialog_idx, "topic", 0)],
                })
            dd[split_name] = Dataset.from_list(rows)
        final[(lang_code, model_key)] = dd
        print(lang_code, model_key, dd)


fr gemma DatasetDict({
    validation: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 1500
    })
    train: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 12460
    })
})
fr qwen DatasetDict({
    validation: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 1500
    })
    train: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 12460
    })
})
es gemma DatasetDict({
    validation: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'dialog', 'summary', 'to

## Spot-check quality

In [10]:
lang_code = "es"
model_key = "gemma"
split_name = SPLITS_TO_RUN[0]
idxs = random.sample(range(len(final[(lang_code, model_key)][split_name])), 3)
for i in idxs:
    row = final[(lang_code, model_key)][split_name][i]
    print("TOPIC:", row["topic"])
    print("SUMMARY:", row["summary"])
    print(row["dialogue"][:300])
    print()


TOPIC: en el restaurante
SUMMARY: #Person1# y #Person2# están esperando para pedir. Se presentan como estudiantes de la escuela de verano, mencionan de dónde vienen y su experiencia con la escuela de verano.
#Person1#: El servicio es muy lento aquí. He estado intentando llamar la atención del camarero durante los últimos diez minutos.
#Person2#: Espero que tome nuestro pedido pronto. De lo contrario, llegaré tarde a mi clase de las dos.
#Person1#: Yo también. También tengo una clase a las 2.
#Person2#: 

TOPIC: pedir prestado un libro
SUMMARY: #Person1# llama a Martin y discuten los exámenes de la próxima semana. Luego, #Person1# le pregunta a Martin sobre su libro de física, y Martin le dice que puede prestarle el suyo. Se reunirán en el almuerzo para intercambiar el libro y los exámenes antiguos.
#Person1#: Hola, Martin. ¿Cómo estás?
#Person2#: Bien, pero ocupado. Tenemos algunos exámenes la próxima semana, ¿recuerdas?
#Person1#: Lo sé. ¿Cuánto trabajaste anoche?
#Person2#: Nada. Fui

## Save

Saves one directory per (language, model). Pushing to the Hub is left commented out --
uncomment and set your own repo id if you want to publish.


In [12]:
SAVE_DIR = Path("translations/dialogsum_final")
for (lang_code, model_key), dd in final.items():
    dd.save_to_disk(str(SAVE_DIR / f"{lang_code}-{model_key}"))

repo_id = "DeepPavlov/dialogsum-translated"
for (lang_code, model_key), dd in final.items():
    dd.push_to_hub(repo_id, config_name=f"{lang_code}-{model_key}")


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12460 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

...

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            